In [8]:
import os
import sys
import csv
import glob
import pandas as pd
from statistics import mean

import pandas as pd
import os

pred_method = "openfold3"

original_directory = "/media/emel/d/slim_af2.3/NEW/AF2_v23_pred/original_pdbs"  ## Native PDB directory
folder_path = f"/media/emel/d/slim_af2.3/NEW/openfold3/openfold_predictions/1st_round"

# For OpenFold3, get top-level directories
complex_list2 = [
    f for f in os.listdir(folder_path)
    if os.path.isdir(os.path.join(folder_path, f)) and not f.startswith(".")
]

complex_list = [f.replace(".result", "") for f in complex_list2]
print(complex_list)

## create results folder
result_folder = pred_method + "_results"
result_path = os.path.join("/media/emel/d/slim_af2.3/NEW/openfold3/", result_folder)
os.makedirs(result_path, exist_ok=True)

# Ensure DockQ is in your path or installed
try:
    from DockQ.DockQ import load_PDB, run_on_all_native_interfaces
except ImportError:
    print("Error: Could not import DockQ. Please ensure the DockQ repository is in your PYTHONPATH.")
    sys.exit(1)

# --- CONFIGURATION ---
ROOT_INPUT_FOLDER = folder_path
ORIGINAL_PDB_FOLDER = original_directory

FINAL_SUMMARY_FILE = "all_dockq_scores.csv"
ERROR_LOG_FILE = "error_log.txt"
# ---------------------

def log_error(message, error_file_path):
    """Appends an error message to the log file."""
    print(f"XXX ERROR: {message}")
    with open(error_file_path, "a") as f:
        f.write(f"{message}\n")

def merge_chains(model, chains_to_merge):
    """Merges specified chains in the given model."""
    for chain in chains_to_merge[1:]:
        for res in list(model[chain]):
            res.id = (chains_to_merge[0], res.id[1], res.id[2])
            model[chains_to_merge[0]].add(res)
        model.detach_child(chain)
    model[chains_to_merge[0]].id = "".join(chains_to_merge)
    return model

def calculate_dockq(model, native, chain_map):
    """Calculates DockQ scores."""
    try:
        results, dockq_score = run_on_all_native_interfaces(model, native, chain_map=chain_map)
        return results, dockq_score
    except Exception as e:
        raise RuntimeError(f"DockQ internal calculation error: {e}")

def process_models(models, error_log_path):
    """Processes the provided models and calculates DockQ scores."""
    results_list = []
    
    for model_file, native_file in models:
        model_id = os.path.basename(model_file)
        native_id = os.path.basename(native_file)
        
        print(f"Processing model: {model_id}")
        
        try:
            model = load_PDB(model_file)
            native = load_PDB(native_file)
        except Exception as e:
            log_error(f"{model_id}: Failed to load PDB files. Error: {e}", error_log_path)
            continue

        chain_ids = list(model.child_dict.keys())
        native_chain_ids = list(native.child_dict.keys())

        try:
            # --- Logic for 3 chains (Merge first two) ---
            if len(chain_ids) == 3:
                if len(native_chain_ids) < 3:
                    raise ValueError(f"Model has 3 chains {chain_ids} but Native {native_id} only has {native_chain_ids}")

                model_merged = merge_chains(model, chain_ids[:2])
                native_merged = merge_chains(native, native_chain_ids[:2])
                
                curr_model_chains = list(model_merged.child_dict.keys())
                curr_native_chains = list(native_merged.child_dict.keys())
                
                if len(curr_native_chains) < 2:
                    raise ValueError(f"Native structure chain merge failed. Resulting chains: {curr_native_chains}")

                chain_map_merged = {curr_native_chains[1]: curr_model_chains[1], curr_native_chains[0]: curr_model_chains[0]}
                
                results_merged, _ = calculate_dockq(model_merged, native_merged, chain_map_merged)
                
                if results_merged:
                    merged_result = results_merged[list(results_merged.keys())[0]]
                    results_list.append((
                        model_id, merged_result['DockQ'], merged_result['fnat'],
                        merged_result['iRMSD'], merged_result['LRMSD'], merged_result['F1']
                    ))

            # --- Logic for 2 chains ---
            elif len(chain_ids) == 2:
                if len(native_chain_ids) < 2:
                    raise ValueError(f"Model has 2 chains {chain_ids} but Native {native_id} only has {native_chain_ids}")
                
                try:
                    target_native_chain_1 = native_chain_ids[0]
                    target_native_chain_2 = native_chain_ids[1]
                except IndexError:
                    raise ValueError(f"Native {native_id} missing chains at index 0 or 1.")

                chain_map = {target_native_chain_1: chain_ids[0], target_native_chain_2: chain_ids[1]}
                
                results, _ = calculate_dockq(model, native, chain_map)
                
                if results:
                    first_key = list(results.keys())[0]
                    results_list.append((
                        model_id, results[first_key]['DockQ'], results[first_key]['fnat'],
                        results[first_key]['iRMSD'], results[first_key]['LRMSD'], results[first_key]['F1']
                    ))
            
            else:
                log_error(f"{model_id}: Skipping. Model has {len(chain_ids)} chains (only 2 or 3 supported).", error_log_path)

        except KeyError as e:
            log_error(f"{model_id}: Chain ID Error. Missing chain in native or model. Details: {e}", error_log_path)
        except ValueError as e:
            log_error(f"{model_id}: Structure Mismatch. {e}", error_log_path)
        except Exception as e:
            log_error(f"{model_id}: Unexpected Error during processing. {e}", error_log_path)

    return results_list

def save_results_to_csv(results, filename):
    if not results:
        return
    print(f"Saving results to: {filename}")
    with open(filename, mode='w', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(['model_id', 'DockQ', 'fnat', "iRMSD", "LRMSD", "F1"])
        for row in results:
            writer.writerow(row)

def process_folder(current_dir, original_directory, error_log_path):
    """
    Modified to handle OpenFold3 structure:
    PDB_ID/PDB_ID/seed_*/PDB_ID_seed_*_sample_*_model.pdb
    """
    # Look for OpenFold3 model files: *_seed_*_sample_*_model.pdb
    pdb_files = glob.glob(os.path.join(current_dir, "*_seed_*_sample_*_model.pdb"))
    
    if not pdb_files:
        return None

    models = []
    base_name_for_csv = ""

    # Match with Natives
    for pdb_file in pdb_files:
        pdb_filename = os.path.basename(pdb_file)
        
        # Parse ID from OpenFold3 naming: "9VNX_seed_1181241943_sample_1_model.pdb"
        # Extract everything before "_seed_"
        pdb_id_base = pdb_filename.split('_seed_')[0]
        
        # Modified to look for _PS.pdb suffix
        native_candidate = os.path.join(original_directory, f"{pdb_id_base}_PS.pdb")
        
        if os.path.exists(native_candidate):
            models.append((pdb_file, native_candidate))
            base_name_for_csv = pdb_id_base
        else:
            log_error(f"{pdb_filename}: Native file not found at {native_candidate}", error_log_path)

    if not models:
        return None

    # Run Calculations
    results = process_models(models, error_log_path)

    # Save Local CSV
    if results:
        csv_name = f"{base_name_for_csv}_dockq_scores.csv" if base_name_for_csv else "dockq_scores.csv"
        output_path = os.path.join(current_dir, csv_name)
        save_results_to_csv(results, output_path)
        return output_path
    
    return None

def main():
    print(f"Starting Scan in: {ROOT_INPUT_FOLDER}")
    print(f"Using Natives from: {ORIGINAL_PDB_FOLDER}")
    
    error_log_path = os.path.join(ROOT_INPUT_FOLDER, ERROR_LOG_FILE)
    # Clear previous error log
    with open(error_log_path, "w") as f:
        f.write("--- DockQ Processing Error Log ---\n")

    all_csv_files = []

    # Walk through all subdirectories (including nested seed folders)
    for root, dirs, files in os.walk(ROOT_INPUT_FOLDER):
        # Check if this directory contains OpenFold3 model files
        if any("_seed_" in f and "_sample_" in f and "_model.pdb" in f for f in files):
            print(f"\n--- Processing Folder: {root} ---")
            created_csv = process_folder(root, ORIGINAL_PDB_FOLDER, error_log_path)
            if created_csv:
                all_csv_files.append(created_csv)

    # Combine all results
    print("\n--- Combining Results ---")
    if all_csv_files:
        df_list = []
        for file in all_csv_files:
            try:
                df = pd.read_csv(file)
                df['Source_Folder'] = os.path.dirname(file)
                df_list.append(df)
            except pd.errors.EmptyDataError:
                pass
        
        if df_list:
            combined_df = pd.concat(df_list, ignore_index=True)
            output_file = os.path.join(ROOT_INPUT_FOLDER, FINAL_SUMMARY_FILE)
            combined_df.to_csv(output_file, index=False)
            print(f"Success! Combined scores saved to: {output_file}")
        else:
            print("No valid data found in CSVs.")
    else:
        print("No results generated.")
        
    print(f"\nProcessing Complete. Check {error_log_path} for any failed files.")

if __name__ == "__main__":
    main()

['8BIA', '8JJS', '8S6U', '7TT8', '9VEW', '9D34', '8AI4', '7YXN', '8SDX', '9KHC', '8QLG', '7TRB', '9CL8', '9M4R', '8OKF', '8Q26', '7T2X', '8YTH', '7QNS', '8JPF', '9KD5', '7YKH', '7X70', '8JOY', '8QQF', '8P0Q', '8WXQ', '8HLO', '8S1R', '7YXR', '9VN5', '8B8O', '8YTG', '7YC2', '8QCI', '7WHU', '8GZE', '8QT3', '8TGP', '7QS9', '8SGF', '8HLW', '9NLO', '8S3M', '8RZU', '8TTD', '7T1U', '8ZE8', '8OIO', '7QF9', '8Q1N', '7WFY', '8CD8', '9G4A', '9HGC', '9DYA', '9NK9', '9J0K', '8PII', '8B58', '8ARE', '7ZPY', '7YMK', '9LDS', '7UO8', '7QPI', '8FUD', '7UDJ', '7UOA', '8HE3', '9M10', '7TB1', '8R7U', '7X8F', '7QS8', '9FIW', '9JGL', '8TTE', '8ACK', '8T32', '8Q2Z', '7YUZ', '7YKG', '7QRS', '8J5U', '8JOW', '7UY2', '9GFA', '8OPI', '8AEL', '7TRL', '8IN0', '8A5L', '9VG3', '8YFK', '8UK5', '9QX6', '8TFU', '8C3H', '8CZK', '8UHL', '9UPZ', '7QTU', '8CZ9', '7QHK', '7TUQ', '7YXP', '8SOU', '9IWN', '8TI7', '8DAF', '8PP0', '7UMA', '8GCW', '7WUL', '7R1V', '7ZFG', '7ZB0', '9VG0', '9KD4', '7YX8', '9KFO', '8R7S', '8CV4', '8WXX',

In [10]:
import pandas as pd

def classify_dockq(score):
    if 0.00 <= score < 0.23:
        return 'Incorrect'
    elif 0.23 <= score < 0.49:
        return 'Acceptable'
    elif 0.49 <= score < 0.80:
        return 'Medium'
    elif score >= 0.80:
        return 'High'
    else:
        return 'Invalid score'

def classify_capri(row):
    fnat, i_rmsd, l_rmsd = row['fnat'], row['iRMSD'], row['LRMSD']

    # High: fnat ≥ 0.5 AND (L-RMSD ≤ 1.0 OR i-RMSD ≤ 1.0)
    if fnat >= 0.5 and (l_rmsd <= 1.0 or i_rmsd <= 1.0):
        return "High"

    # Medium: fnat ≥ 0.3 AND (L-RMSD ≤ 5.0 OR i-RMSD ≤ 2.0)
    if fnat >= 0.3 and (l_rmsd <= 5.0 or i_rmsd <= 2.0):
        return "Medium"

    # Acceptable: fnat ≥ 0.1 AND (L-RMSD ≤ 10.0 OR i_RMSD ≤ 4.0)
    if fnat >= 0.1 and (l_rmsd <= 10.0 or i_rmsd <= 4.0):
        return "Acceptable"

    # Otherwise, Incorrect
    return "Incorrect"

def classify_capri_peptide(row):
    fnat, i_rmsd, l_rmsd = row['fnat'], row['iRMSD'], row['LRMSD']

    # High: fnat ∈ [0.8, 1.0] AND (L-RMSD ≤ 1.0 OR i-RMSD ≤ 0.5)
    if 0.8 <= fnat <= 1.0 and (l_rmsd <= 1.0 or i_rmsd <= 0.5):
        return "High"

    # Medium:
    # Case 1: fnat ∈ [0.5, 0.8] AND (L-RMSD ≤ 2.0 OR i-RMSD ≤ 1.0)
    # OR Case 2: fnat ∈ [0.8, 1.0] AND (L-RMSD > 1.0 AND i-RMSD > 0.5)
    if (0.5 <= fnat < 0.8 and (l_rmsd <= 2.0 or i_rmsd <= 1.0)) or \
       (0.8 <= fnat <= 1.0 and (l_rmsd > 1.0 and i_rmsd > 0.5)):
        return "Medium"

    # Acceptable:
    # Case 1: fnat ∈ [0.2, 0.5] AND (L-RMSD ≤ 4.0 OR i-RMSD ≤ 2.0)
    # OR Case 2: fnat ∈ [0.5, 1.0] AND (L-RMSD > 2.0 AND i-RMSD > 1.0)
    if (0.2 <= fnat < 0.5 and (l_rmsd <= 4.0 or i_rmsd <= 2.0)) or \
       (0.5 <= fnat <= 1.0 and (l_rmsd > 2.0 and i_rmsd > 1.0)):
        return "Acceptable"

    # All else: Incorrect
    return "Incorrect"
df=pd.read_csv("/media/emel/d/slim_af2.3/NEW/openfold3/openfold_predictions/all_dockq_scores.csv")
print(list(df))
df['CAPRI'] = df.apply(classify_capri, axis=1)
df['CAPRIp'] = df.apply(classify_capri_peptide, axis=1)
df['DockQ_category'] = df['DockQ'].apply(classify_dockq)
# df["model_confidence"] = 0.8*df["ipTM"]+0.2 *df["pTM"]

print(list(df))
df.to_csv(f"{result_path}/{dockq_output}")
df

['model_id', 'DockQ', 'fnat', 'iRMSD', 'LRMSD', 'F1', 'Source_Folder']
['model_id', 'DockQ', 'fnat', 'iRMSD', 'LRMSD', 'F1', 'Source_Folder', 'CAPRI', 'CAPRIp', 'DockQ_category']


,model_id,DockQ,fnat,iRMSD,LRMSD,F1,Source_Folder,CAPRI,CAPRIp,DockQ_category
0,8BIA_seed_1812140441_sample_3_model.pdb,0.163629,0.214286,5.683865,16.412618,0.236842,/media/emel/d/slim_af2.3/NEW/openfold3/openfol...,Incorrect,Incorrect,Incorrect
1,8BIA_seed_1812140441_sample_5_model.pdb,0.119239,0.119048,6.128530,18.011090,0.140845,/media/emel/d/slim_af2.3/NEW/openfold3/openfol...,Incorrect,Incorrect,Incorrect
2,8BIA_seed_1812140441_sample_2_model.pdb,0.149615,0.142857,5.326835,15.442721,0.164384,/media/emel/d/slim_af2.3/NEW/openfold3/openfol...,Incorrect,Incorrect,Incorrect
3,8BIA_seed_1812140441_sample_1_model.pdb,0.151283,0.142857,5.249432,15.314634,0.169014,/media/emel/d/slim_af2.3/NEW/openfold3/openfol...,Incorrect,Incorrect,Incorrect
4,8BIA_seed_1812140441_sample_4_model.pdb,0.162454,0.190476,5.395038,15.769384,0.202532,/media/emel/d/slim_af2.3/NEW/openfold3/openfol...,Incorrect,Incorrect,Incorrect
...,...,...,...,...,...,...,...,...,...,...
7320,8EBL_seed_2746317213_sample_3_model.pdb,0.170305,0.068182,3.605434,13.134365,0.101695,/media/emel/d/slim_af2.3/NEW/openfold3/openfol...,Incorrect,Incorrect,Incorrect
7321,8EBL_seed_2746317213_sample_1_model.pdb,0.121949,0.000000,4.043638,14.925980,0.000000,/media/emel/d/slim_af2.3/NEW/openfold3/openfol...,Incorrect,Incorrect,Incorrect
7322,8EBL_seed_2746317213_sample_4_model.pdb,0.122249,0.090909,4.738127,17.855906,0.140351,/media/emel/d/slim_af2.3/NEW/openfold3/openfol...,Incorrect,Incorrect,Incorrect
7323,8EBL_seed_2746317213_sample_5_model.pdb,0.095718,0.022727,4.832775,18.356658,0.041667,/media/emel/d/slim_af2.3/NEW/openfold3/openfol...,Incorrect,Incorrect,Incorrect


In [18]:
import os
import glob
import pandas as pd
import json
from Bio.PDB import PDBIO
from Bio.PDB.PDBParser import PDBParser
from Bio.PDB.Selection import unfold_entities
import numpy as np
import sys
import argparse
import pickle
import itertools

### Interface score extraction functions

def retrieve_IFplddt(structure, chain1, chain2_lst, max_dist):
    """
    Extract interface pLDDT scores for residues in contact
    Returns per-chain average, all values, and contact chains
    """
    chain_lst = list(chain1) + chain2_lst
    ifplddt = []
    contact_chain_lst = []
    for res1 in structure[0][chain1]:
        for chain2 in chain2_lst:
            count = 0
            for res2 in structure[0][chain2]:
                if res1.has_id('CA') and res2.has_id('CA'):
                   dis = abs(res1['CA']-res2['CA'])
                   ## add criteria to filter out disorder res
                   if dis <= max_dist:
                      ifplddt.append(res1['CA'].get_bfactor())
                      count += 1
                elif res1.has_id('CB') and res2.has_id('CB'):
                   dis = abs(res1['CB']-res2['CB'])
                   if dis <= max_dist:
                      ifplddt.append(res1['CB'].get_bfactor())
                      count += 1
            if count > 0:
              contact_chain_lst.append(chain2)
    contact_chain_lst = sorted(list(set(contact_chain_lst)))   
    if len(ifplddt)>0:
       IF_plddt_avg = np.mean(ifplddt)
    else:
       IF_plddt_avg = 0
    return IF_plddt_avg, ifplddt, contact_chain_lst


def retrieve_IFPDEinter(structure, pdeMat, contact_lst, max_dist):
    """
    Extract raw interface PDE (PAE) scores WITHOUT normalization
    Returns per-chain averages and all values
    """
    chain_lst = [x.id for x in structure[0]]
    seqlen = [len(x) for x in structure[0]]
    ifpde_per_chain = {}  # Dictionary to store per-chain scores
    ifpde_all = []  # Store all individual PDE values
    
    for ch1_idx in range(len(chain_lst)):
        chain_id = chain_lst[ch1_idx]
        idx = chain_lst.index(chain_id)
        ch1_sta = sum(seqlen[:idx])
        ch1_end = ch1_sta + seqlen[idx]
        ifpde_col = []   
        
        for contact_ch in contact_lst[ch1_idx]:
            index = chain_lst.index(contact_ch)
            ch_sta = sum(seqlen[:index])
            ch_end = ch_sta + seqlen[index]
            pdeMat = np.array(pdeMat)
            remain_pdeMatrix = pdeMat[ch1_sta:ch1_end, ch_sta:ch_end]
            mat_x = -1
            
            for res1 in structure[0][chain_id]:
                mat_x += 1
                mat_y = -1
                for res2 in structure[0][contact_ch]:
                    mat_y += 1
                    if res1['CA'] - res2['CA'] <= max_dist:
                        ifpde_col.append(remain_pdeMatrix[mat_x, mat_y])
        
        # Store per-chain average
        if not ifpde_col:
            ifpde_per_chain[chain_id] = 0
        else:
            ifpde_per_chain[chain_id] = np.mean(ifpde_col)
            ifpde_all.extend(ifpde_col)
    
    return ifpde_per_chain, ifpde_all

def process_pdb_file(pdb_file, json_file, distance, file_id, chains_part=""): 
    """
    Process PDB and extract raw interface PDE and pLDDT scores per chain
    """
    pdbp = PDBParser(QUIET=True)
    structure = pdbp.get_structure('', pdb_file)
    chains = [chain.id for chain in structure[0]]
    remain_contact_lst = []
    plddt_per_chain = {}
    all_ifplddt = []
    
    # Get interface pLDDT for each chain
    for idx in range(len(chains)):
        chain_id = chains[idx]
        chain2_lst = list(set(chains) - set(chain_id))
        IF_plddt, ifplddt_vals, contact_lst = retrieve_IFplddt(structure, chain_id, chain2_lst, distance)
        plddt_per_chain[chain_id] = IF_plddt
        all_ifplddt.extend(ifplddt_vals)
        remain_contact_lst.append(contact_lst)
    
    # Load PDE matrix from JSON
    with open(json_file, 'r') as f:
        pde_data = json.load(f)
    
    # OpenFold3 uses "pde" instead of "pae"
    pde_matrix = pde_data.get("pde", pde_data.get("pae", None))
    
    if pde_matrix is None:
        raise ValueError(f"Neither 'pde' nor 'pae' found in {json_file}")
    
    # Get interface PDE scores per chain (raw, no normalization)
    ifpde_per_chain, ifpde_all = retrieve_IFPDEinter(structure, pde_matrix, remain_contact_lst, distance)
    
    pdb_id = os.path.basename(pdb_file).split('_')[0]
    
    # Build result dictionary with both per-chain and overall statistics
    result = {
        "model_id": os.path.basename(pdb_file),
        "pdb_id": file_id,
        "pdb_id_with_chains": '{0}_{1}'.format(pdb_id, chains_part) if chains_part else pdb_id,
        "chains": "_".join(sorted(chains)),
    }
    
    # Add per-chain interface PDE scores
    for chain_id, pde_score in ifpde_per_chain.items():
        result[f"interface_pde_chain_{chain_id}"] = pde_score
    
    # Add per-chain interface pLDDT scores
    for chain_id, plddt_score in plddt_per_chain.items():
        result[f"interface_plddt_chain_{chain_id}"] = plddt_score
    
    # Add overall statistics for interface PDE
    result.update({
        "interface_pde_avg": np.mean(list(ifpde_per_chain.values())) if ifpde_per_chain else 0,
        "interface_pde_overall": np.mean(ifpde_all) if ifpde_all else 0,
        "interface_pde_min": np.min(ifpde_all) if ifpde_all else 0,
        "interface_pde_max": np.max(ifpde_all) if ifpde_all else 0,
        "interface_pde_std": np.std(ifpde_all) if ifpde_all else 0,
        "interface_pde_median": np.median(ifpde_all) if ifpde_all else 0,
    })
    
    # Add overall statistics for interface pLDDT
    result.update({
        "interface_plddt_avg": np.mean(list(plddt_per_chain.values())) if plddt_per_chain else 0,
        "interface_plddt_overall": np.mean(all_ifplddt) if all_ifplddt else 0,
        "interface_plddt_min": np.min(all_ifplddt) if all_ifplddt else 0,
        "interface_plddt_max": np.max(all_ifplddt) if all_ifplddt else 0,
        "interface_plddt_std": np.std(all_ifplddt) if all_ifplddt else 0,
        
        # Number of interface contacts
        "n_interface_contacts": len(ifpde_all)
    })
    
    return result

def find_matching_json(pdb_file, directory):
    """
    For OpenFold3 predictions:
    PDB: 9VNX_seed_1181241943_sample_1_model.pdb
    JSON: 9VNX_seed_1181241943_sample_1_confidences.json
    """
    pdb_file_basename = os.path.basename(pdb_file)
    
    # Replace "_model.pdb" with "_confidences.json"
    json_filename = pdb_file_basename.replace("_model.pdb", "_confidences.json")
    json_path = os.path.join(directory, json_filename)
    
    if os.path.exists(json_path):
        return json_path
    else:
        print(f"JSON file not found: {json_path}")
        return None

def remove_existing_csvs(folder_path, pattern="*_pdockq2_scores.csv"):
    """
    Remove or overwrite existing pDockQ2 CSV files
    """
    csv_files = glob.glob(os.path.join(folder_path, "**/" + pattern), recursive=True)
    
    if csv_files:
        print(f"\nFound {len(csv_files)} existing CSV files to remove:")
        for csv_file in csv_files:
            try:
                os.remove(csv_file)
                print(f"  ✓ Removed: {csv_file}")
            except Exception as e:
                print(f"  ✗ Error removing {csv_file}: {e}")
    else:
        print("\nNo existing CSV files found to remove.")

def run_processing(current_dir, result_output_path):
    """
    Modified for OpenFold3 structure:
    Looks for: *_seed_*_sample_*_model.pdb
    """
    # Search for OpenFold3 PDB files
    pdb_files = glob.glob(os.path.join(current_dir, "*_seed_*_sample_*_model.pdb"))
    
    if not pdb_files:
        return None
    
    results = []    
    current_file_id = "unknown_complex"
    
    for pdb_file in pdb_files:
        pdb_file_basename = os.path.basename(pdb_file)
        
        # Parse ID from OpenFold3 naming: "9VNX_seed_1181241943_sample_1_model.pdb"
        # Extract PDB ID (everything before "_seed_")
        pdb_id = pdb_file_basename.split('_seed_')[0]
        
        # Extract seed and sample info for chains_part (optional)
        parts = pdb_file_basename.replace("_model.pdb", "").split('_')
        # chains_part could be seed_XXXX_sample_Y or just empty
        chains_part = "_".join(parts[1:]) if len(parts) > 1 else ""
        
        json_file_name = find_matching_json(pdb_file, current_dir)
        
        if json_file_name and os.path.exists(json_file_name):
            print(f"Processing: {pdb_file_basename}")
            
            # Update file_id for naming the CSV later
            current_file_id = pdb_id
            
            try:
                result = process_pdb_file(pdb_file, json_file_name, 8, pdb_id, chains_part) 
                
                # Print results with chain B info if available
                chain_b_pde = result.get('interface_pde_chain_B', 'N/A')
                chain_b_plddt = result.get('interface_plddt_chain_B', 'N/A')
                
                if chain_b_pde != 'N/A':
                    print(f"  Chain B - PDE: {chain_b_pde:.3f}, pLDDT: {chain_b_plddt:.2f}")
                print(f"  Overall - PDE avg: {result['interface_pde_avg']:.3f}, pLDDT avg: {result['interface_plddt_avg']:.2f}")
                
                results.append(result)
            except Exception as e:
                print(f"Error processing {pdb_file}: {e}")
                continue
        else:
            print(f"JSON file for {pdb_file} not found.")
            continue

    if results:
        df = pd.DataFrame(results)
        
        csv_filename = f"{current_file_id}_interface_scores.csv"
        csv_file_path = os.path.join(current_dir, csv_filename)
        
        df.to_csv(csv_file_path, index=False)
        print(f"Data saved to {csv_file_path}")
        return csv_file_path
    else:
        print(f"No results generated for {current_dir}")
        return None

def combine_csv_files(result_path, output_file=None):
    """Combine all individual CSV files into one summary file"""
    csv_files = glob.glob(os.path.join(result_path, "**/*_interface_scores.csv"), recursive=True)
    
    if not csv_files:
        print("No CSV files found to combine.")
        return pd.DataFrame()

    print(f"Found {len(csv_files)} CSV files to combine")
    df_list = []
    for file in csv_files:
        try:
            df = pd.read_csv(file)
            df['Source_Folder'] = os.path.dirname(file)
            df_list.append(df)
        except Exception as e:
            print(f"Error reading {file}: {e}")
    
    if df_list:
        combined_df = pd.concat(df_list, ignore_index=True)
        
        if output_file:
            output_path = os.path.join(result_path, output_file)
            combined_df.to_csv(output_path, index=False)
            print(f"Combined CSV saved to {output_path}")
        
        return combined_df
    else:
        return pd.DataFrame()

# --- Main Execution ---

if __name__ == "__main__":
    
    pred_method = "openfold3"
    
    # Update these paths to match your setup
    folder_path = "/media/emel/d/slim_af2.3/NEW/openfold3/openfold_predictions/"
    
    # Remove existing pDockQ2 CSV files
    print("="*70)
    print("Cleaning up existing CSV files...")
    print("="*70)
    remove_existing_csvs(folder_path, "*_pdockq2_scores.csv")
    remove_existing_csvs(folder_path, "*_interface_scores.csv")
    
    # Get list of complexes (top-level directories)
    complex_list2 = [
        f for f in os.listdir(folder_path)
        if os.path.isdir(os.path.join(folder_path, f)) and not f.startswith(".")
    ]
    
    complex_list = [f.replace(".result", "") for f in complex_list2]
    print(f"\nFound {len(complex_list)} complexes to process")
    print(complex_list)
    
    # Create results folder
    result_folder = pred_method + "_results"
    result_path = os.path.join("/media/emel/d/slim_af2.3/NEW/openfold3/", result_folder)
    os.makedirs(result_path, exist_ok=True)
    
    interface_output = f"{pred_method}_interface_scores_combined.csv"
    
    print(f"\nStarting batch processing on {len(complex_list)} complexes...")
    print("="*70)
    
    all_csv_files = []
    
    # Walk through all subdirectories (including nested seed folders)
    for root, dirs, files in os.walk(folder_path):
        # Check if this directory contains OpenFold3 model files
        if any("_seed_" in f and "_sample_" in f and "_model.pdb" in f for f in files):
            print(f"\n--- Processing Folder: {root} ---")
            created_csv = run_processing(root, result_path)
            if created_csv:
                all_csv_files.append(created_csv)
    
    # Combine all results
    print("\n" + "="*70)
    print("Combining Results")
    print("="*70)
    combined_df = combine_csv_files(folder_path, interface_output)
    
    if not combined_df.empty:
        print(f"\n✓ Success! Processed {len(combined_df)} models")
        print(f"✓ Combined results saved to: {os.path.join(folder_path, interface_output)}")
        
        # Print summary statistics
        # print("\nSummary Statistics:")
        # print(f"  Average Interface PDE (overall): {combined_df['interface_pde_avg'].mean():.3f} ± {combined_df['interface_pde_avg'].std():.3f}")
        # print(f"  Average Interface pLDDT (overall): {combined_df['interface_plddt_avg'].mean():.2f} ± {combined_df['interface_plddt_avg'].std():.2f}")
        # print(f"  Average Interface Contacts: {combined_df['n_interface_contacts'].mean():.1f}")
        
        # # Show chain-specific statistics if available
        # if 'interface_pde_chain_B' in combined_df.columns:
        #     print(f"\n  Chain B Interface PDE: {combined_df['interface_pde_chain_B'].mean():.3f} ± {combined_df['interface_pde_chain_B'].std():.3f}")
        #     print(f"  Chain B Interface pLDDT: {combined_df['interface_plddt_chain_B'].mean():.2f} ± {combined_df['interface_plddt_chain_B'].std():.2f}")
        
        # if 'interface_pde_chain_A' in combined_df.columns:
        #     print(f"  Chain A Interface PDE: {combined_df['interface_pde_chain_A'].mean():.3f} ± {combined_df['interface_pde_chain_A'].std():.3f}")
        #     print(f"  Chain A Interface pLDDT: {combined_df['interface_plddt_chain_A'].mean():.2f} ± {combined_df['interface_plddt_chain_A'].std():.2f}")
    else:
        print("\n✗ No results generated.")
    
    # print("\n" + "="*70)
    # print("Processing Complete!")
    # print("="*70)

Cleaning up existing CSV files...

No existing CSV files found to remove.

Found 93 existing CSV files to remove:
  ✓ Removed: /media/emel/d/slim_af2.3/NEW/openfold3/openfold_predictions/8BIA/8BIA/seed_1812140441/8BIA_interface_scores.csv
  ✓ Removed: /media/emel/d/slim_af2.3/NEW/openfold3/openfold_predictions/8BIA/8BIA/seed_1181241943/8BIA_interface_scores.csv
  ✓ Removed: /media/emel/d/slim_af2.3/NEW/openfold3/openfold_predictions/8BIA/8BIA/seed_3163119785/8BIA_interface_scores.csv
  ✓ Removed: /media/emel/d/slim_af2.3/NEW/openfold3/openfold_predictions/8BIA/8BIA/seed_958682846/8BIA_interface_scores.csv
  ✓ Removed: /media/emel/d/slim_af2.3/NEW/openfold3/openfold_predictions/8BIA/8BIA/seed_2746317213/8BIA_interface_scores.csv
  ✓ Removed: /media/emel/d/slim_af2.3/NEW/openfold3/openfold_predictions/8JJS/8JJS/seed_1812140441/8JJS_interface_scores.csv
  ✓ Removed: /media/emel/d/slim_af2.3/NEW/openfold3/openfold_predictions/8JJS/8JJS/seed_1181241943/8JJS_interface_scores.csv
  ✓ Removed:

In [15]:
import os
import glob
import json
import pandas as pd

def extract_aggregated_scores(json_file):
    """
    Extract scores from OpenFold3 aggregated confidence JSON file
    """
    with open(json_file, 'r') as f:
        data = json.load(f)
    
    # Extract main scores
    scores = {
        'avg_plddt': data.get('avg_plddt', None),
        'gpde': data.get('gpde', None),
        'iptm': data.get('iptm', None),
        'ptm': data.get('ptm', None),
        'disorder': data.get('disorder', None),
        'has_clash': data.get('has_clash', None),
        'sample_ranking_score': data.get('sample_ranking_score', None)
    }
    
    # Extract chain_ptm scores (as separate columns)
    chain_ptm = data.get('chain_ptm', {})
    for chain, score in chain_ptm.items():
        scores[f'chain_ptm_{chain}'] = score
    
    # Extract chain_pair_iptm scores
    chain_pair_iptm = data.get('chain_pair_iptm', {})
    for chain_pair, score in chain_pair_iptm.items():
        # Clean up the chain pair notation: "(A, B)" -> "A_B"
        clean_pair = chain_pair.replace('(', '').replace(')', '').replace(', ', '_').replace(' ', '')
        scores[f'chain_pair_iptm_{clean_pair}'] = score
    
    # Extract bespoke_iptm scores
    bespoke_iptm = data.get('bespoke_iptm', {})
    for chain_pair, score in bespoke_iptm.items():
        clean_pair = chain_pair.replace('(', '').replace(')', '').replace(', ', '_').replace(' ', '')
        scores[f'bespoke_iptm_{clean_pair}'] = score
    
    return scores

def find_aggregated_json(pdb_file):
    """
    Find the corresponding aggregated JSON file for a PDB file
    PDB: 7QF9_seed_958682846_sample_1_model.pdb
    JSON: 7QF9_seed_958682846_sample_1_confidences_aggregated.json
    """
    directory = os.path.dirname(pdb_file)
    pdb_basename = os.path.basename(pdb_file)
    
    # Replace "_model.pdb" with "_confidences_aggregated.json"
    json_filename = pdb_basename.replace("_model.pdb", "_confidences_aggregated.json")
    json_path = os.path.join(directory, json_filename)
    
    if os.path.exists(json_path):
        return json_path
    else:
        print(f"Aggregated JSON not found: {json_path}")
        return None

def process_openfold3_directory(root_folder):
    """
    Walk through OpenFold3 directory and extract all aggregated scores
    """
    results = []
    
    # Find all PDB model files
    print(f"Scanning directory: {root_folder}")
    
    for root, dirs, files in os.walk(root_folder):
        # Look for OpenFold3 model files
        pdb_files = [f for f in files if f.endswith("_model.pdb") and "_seed_" in f and "_sample_" in f]
        
        if pdb_files:
            print(f"\nProcessing folder: {root}")
            print(f"Found {len(pdb_files)} PDB files")
            
            for pdb_file in pdb_files:
                pdb_path = os.path.join(root, pdb_file)
                
                # Find corresponding aggregated JSON
                json_path = find_aggregated_json(pdb_path)
                
                if json_path:
                    try:
                        # Extract scores
                        scores = extract_aggregated_scores(json_path)
                        
                        # Parse PDB filename to extract metadata
                        pdb_basename = os.path.basename(pdb_file)
                        parts = pdb_basename.replace("_model.pdb", "").split('_')
                        
                        pdb_id = parts[0] if parts else "unknown"
                        
                        # Find seed and sample numbers
                        seed = None
                        sample = None
                        for i, part in enumerate(parts):
                            if part == "seed" and i + 1 < len(parts):
                                seed = parts[i + 1]
                            elif part == "sample" and i + 1 < len(parts):
                                sample = parts[i + 1]
                        
                        # Create result entry
                        result = {
                            'model_file': pdb_basename,
                            'pdb_id': pdb_id,
                            'seed': seed,
                            'sample': sample,
                            'source_folder': root,
                            **scores  # Unpack all the scores
                        }
                        
                        results.append(result)
                        print(f"  ✓ Processed: {pdb_basename} (ranking_score: {scores.get('sample_ranking_score', 'N/A'):.4f})")
                        
                    except Exception as e:
                        print(f"  ✗ Error processing {pdb_file}: {e}")
                else:
                    print(f"  ✗ No aggregated JSON found for {pdb_file}")
    
    return results

def save_results(results, output_file):
    """
    Save results to CSV file
    """
    if not results:
        print("\nNo results to save!")
        return None
    
    df = pd.DataFrame(results)
    
    # Reorder columns to put metadata first
    metadata_cols = ['model_file', 'pdb_id', 'seed', 'sample', 'source_folder']
    score_cols = [col for col in df.columns if col not in metadata_cols]
    df = df[metadata_cols + score_cols]
    
    df.to_csv(output_file, index=False)
    print(f"\n✓ Results saved to: {output_file}")
    print(f"  Total models processed: {len(df)}")
    
    # Print summary statistics
    print("\nSummary Statistics:")
    print(f"  Unique PDB IDs: {df['pdb_id'].nunique()}")
    print(f"  Unique Seeds: {df['seed'].nunique()}")
    print(f"  Samples per complex: {df.groupby('pdb_id').size().mean():.1f}")
    
    if 'avg_plddt' in df.columns:
        print(f"\n  Average pLDDT: {df['avg_plddt'].mean():.2f} ± {df['avg_plddt'].std():.2f}")
    if 'iptm' in df.columns:
        print(f"  Average ipTM: {df['iptm'].mean():.3f} ± {df['iptm'].std():.3f}")
    if 'sample_ranking_score' in df.columns:
        print(f"  Average Ranking Score: {df['sample_ranking_score'].mean():.4f} ± {df['sample_ranking_score'].std():.4f}")
    
    return df

def get_best_models(df, metric='sample_ranking_score', top_n=5):
    """
    Get the best models based on a specific metric
    """
    if df is None or df.empty:
        return None
    
    if metric not in df.columns:
        print(f"Metric '{metric}' not found in dataframe!")
        return None
    
    # Sort by metric (descending)
    best_models = df.nlargest(top_n, metric)[['model_file', 'pdb_id', 'seed', 'sample', metric, 'source_folder']]
    
    print(f"\nTop {top_n} models by {metric}:")
    print(best_models.to_string(index=False))
    
    return best_models

# --- Main Execution ---

if __name__ == "__main__":
    
    # Configuration
    pred_method = "openfold3"
    root_folder = "/media/emel/d/slim_af2.3/NEW/openfold3/openfold_predictions/"
    output_folder = "/media/emel/d/slim_af2.3/NEW/openfold3/"
    
    # Create output directory if it doesn't exist
    os.makedirs(output_folder, exist_ok=True)
    
    output_file = os.path.join(output_folder, f"{pred_method}_aggregated_scores.csv")
    
    print("="*70)
    print("OpenFold3 Aggregated Scores Extraction")
    print("="*70)
    
    # Process all directories
    results = process_openfold3_directory(root_folder)
    
    # Save results
    df = save_results(results, output_file)
    df["model_confidence"] = 0.8*df["iptm"]+0.2 *df["ptm"]
    # Show best models
    if df is not None and not df.empty:
        print("\n" + "="*70)
        get_best_models(df, metric='sample_ranking_score', top_n=10)
        print("="*70)
        
        # Also show best by ipTM if available
        if 'iptm' in df.columns:
            print("\n" + "="*70)
            get_best_models(df, metric='iptm', top_n=10)
            print("="*70)
    
    print("\nProcessing complete!")

df.to_csv(f"{result_path}/openfold3_extracted_metrics.csv", index=False)

OpenFold3 Aggregated Scores Extraction
Scanning directory: /media/emel/d/slim_af2.3/NEW/openfold3/openfold_predictions/

Processing folder: /media/emel/d/slim_af2.3/NEW/openfold3/openfold_predictions/8BIA/8BIA/seed_1812140441
Found 5 PDB files
  ✓ Processed: 8BIA_seed_1812140441_sample_3_model.pdb (ranking_score: 0.9968)
  ✓ Processed: 8BIA_seed_1812140441_sample_5_model.pdb (ranking_score: 1.0067)
  ✓ Processed: 8BIA_seed_1812140441_sample_2_model.pdb (ranking_score: 1.0040)
  ✓ Processed: 8BIA_seed_1812140441_sample_1_model.pdb (ranking_score: 1.0060)
  ✓ Processed: 8BIA_seed_1812140441_sample_4_model.pdb (ranking_score: 1.0170)

Processing folder: /media/emel/d/slim_af2.3/NEW/openfold3/openfold_predictions/8BIA/8BIA/seed_1181241943
Found 5 PDB files
  ✓ Processed: 8BIA_seed_1181241943_sample_5_model.pdb (ranking_score: 1.0051)
  ✓ Processed: 8BIA_seed_1181241943_sample_3_model.pdb (ranking_score: 1.0070)
  ✓ Processed: 8BIA_seed_1181241943_sample_2_model.pdb (ranking_score: 0.9992)
